<a href="https://colab.research.google.com/github/Faizaa01/Deep_Learning_Practices/blob/main/Backword_propagation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
import torch

data = torch.tensor(
    [
        [2,3,3],
        [5,6,4],
        [7,8,5]
    ]
)
data

tensor([[2, 3, 3],
        [5, 6, 4],
        [7, 8, 5]])

In [15]:
X = data[:,0:2].float()
y = data[:,2].float()
X,y

(tensor([[2., 3.],
         [5., 6.],
         [7., 8.]]),
 tensor([3., 4., 5.]))

weight initialization

In [16]:
def initializa_parameters():
  parameters = {}

  parameters['W1'] = torch.ones((2,2)) * 0.1
  parameters['b1'] = torch.zeros((2,1))

  parameters['W2'] = torch.ones((2,1)) * 0.1
  parameters['b2'] = torch.zeros((1,1))

  return parameters

initializa_parameters()

{'W1': tensor([[0.1000, 0.1000],
         [0.1000, 0.1000]]),
 'b1': tensor([[0.],
         [0.]]),
 'W2': tensor([[0.1000],
         [0.1000]]),
 'b2': tensor([[0.]])}

In [24]:
def initialize_parameters(layer_dims):
  torch.manual_seed(3)
  parameters = {}
  L = len(layer_dims)

  for l in range(1,L):
    parameters['W' + str(l)] = torch.randn((layer_dims[l-1], layer_dims[l])) * 0.1
    parameters['b' + str(l)] = torch.zeros((layer_dims[l],1))

  return parameters

initializa_parameters([2,2,1])


{'W1': tensor([[ 0.0803,  0.0175],
         [ 0.0089, -0.0614]]),
 'b1': tensor([[0.],
         [0.]]),
 'W2': tensor([[ 0.0046],
         [-0.1368]]),
 'b2': tensor([[0.]])}

In [25]:
def linear_forward(A_prev, W, b):
  z = torch.matmul(W.T, A_prev) + b
  return z

def L_layer_forward(X, parameters):
  A = X
  L = len(parameters) // 2

  for l in range(1,L+1):
    A_prev = A

    wl = parameters['W' + str(l)]
    bl = parameters['b' + str(l)]

    A = linear_forward(A_prev, wl, bl)

  return A, A_prev

In [19]:
X

tensor([[2., 3.],
        [5., 6.],
        [7., 8.]])

In [26]:
params = initialize_parameters([2,2,1])
X_sample = X[0].reshape(2,1)
y_hat, A1 = L_layer_forward(X_sample, params)
y_hat, A1

(tensor([[0.0213]]),
 tensor([[ 0.1873],
         [-0.1491]]))

In [27]:
loss = (y-y_hat)**2
loss

tensor([[ 8.8728, 15.8303, 24.7877]])

In [28]:
params

{'W1': tensor([[ 0.0803,  0.0175],
         [ 0.0089, -0.0614]]),
 'b1': tensor([[0.],
         [0.]]),
 'W2': tensor([[ 0.0046],
         [-0.1368]]),
 'b2': tensor([[0.]])}

In [29]:
def  update_parameters(parameters, y, y_hat, A1, X, lr=0.001):
  err_signal = 2*(y-y_hat)

  W2_00_old = parameters['W2'][0, 0].item()
  W2_10_old = parameters['W2'][1, 0].item()

  parameters['W2'][0, 0] += lr * err_signal * A1[0, 0]
  parameters['W2'][1, 0] += lr * err_signal * A1[1, 0]
  parameters['b2'][0, 0] += lr * err_signal


  parameters['W1'][0, 0] += lr * err_signal * parameters['W2'][0, 0] * X[0, 0]
  parameters['W1'][0, 1] += lr * err_signal * parameters['W2'][0, 0] * X[1, 0]
  parameters['b1'][0, 0] += lr * err_signal * parameters['W2'][0, 0]

  parameters['W1'][1, 0] += lr * err_signal * parameters['W2'][1, 0] * X[0, 0]
  parameters['W1'][1, 1] += lr * err_signal * parameters['W2'][1, 0] * X[1, 0]
  parameters['b1'][1, 0] += lr * err_signal * parameters['W2'][1, 0]

  return parameters

In [30]:
params = initialize_parameters([2, 2, 1])
epochs = 5

for i in range(epochs):
    epoch_loss = 0
    for j in range(X.shape[0]):
        # Prepare sample
        X_sample = X[j].reshape(2, 1)
        y_sample = y[j]

        # Forward
        y_hat, A1 = L_layer_forward(X_sample, params)
        y_hat_scalar = y_hat[0, 0]

        # Update
        params = update_parameters(params, y_sample, y_hat_scalar, A1, X_sample)

        # Loss calculation
        loss = (y_sample - y_hat_scalar) ** 2
        epoch_loss += loss.item()

    print(f"Epoch - {i+1} Loss - {epoch_loss / X.shape[0]}")

Epoch - 1 Loss - 16.226633707682293
Epoch - 2 Loss - 15.86297639211019
Epoch - 3 Loss - 15.495257695515951
Epoch - 4 Loss - 15.106503168741861
Epoch - 5 Loss - 14.678457260131836
